In [ ]:
from datasets import load_dataset

# 注意：请先从 ModelScope 下载数据集
# modelscope download --dataset modelscope/mnist --local_dir ./mnist_data

# 注意：这里的路径可能需要修改成你自己的路径
# 注意：split="train" 是必须的，因为 parquet 文件本身没有 split 信息，我们需要手动指定
dataset = load_dataset(
    "parquet", 
    data_files="./mnist_data/mnist/train-00000-of-00001.parquet",
    split="train"
)

print(f"数据集已加载，共 {len(dataset)} 条样本。")
print(f"列名: {dataset.column_names}")

def preprocess(dataset):
    processed_images = []
    for img in dataset['image']:
        pixel_data = list(img.getdata())
        img_tensor = torch.tensor(pixel_data, dtype=torch.float32) # 将像素转换为 PyTorch Tensor
        normalized_img = img_tensor / 255.0 # 将像素值从 [0, 255] 缩放到 [0, 1]
        processed_images.append(normalized_img)
    images = torch.stack(processed_images)
    labels = torch.tensor(dataset['label'], dtype=torch.long)
    return images, labels

import torch
from torch import nn

def flatten(x):
    new_x = x.reshape(-1, 784)
    return new_x

layer512_weights = nn.Parameter(torch.randn(784, 512, requires_grad=True) * 0.01)
layer512_bias = nn.Parameter(torch.zeros(512, requires_grad=True))

def layer512(x):
    new_x = torch.matmul(x, layer512_weights) + layer512_bias
    return new_x

layer256_weights = nn.Parameter(torch.randn(512, 256, requires_grad=True) * 0.01)
layer256_bias = nn.Parameter(torch.zeros(256, requires_grad=True))

def layer256(x):
    new_x = x @ layer256_weights + layer256_bias # @ 符是 torch.matmul() 的等价简写
    return new_x

def relu(x):
    zero_shaped = torch.zeros_like(x)
    new_x = torch.max(x, zero_shaped)
    return new_x

layer10_weights = nn.Parameter(torch.randn(256, 10, requires_grad=True) * 0.01)
layer10_bias = nn.Parameter(torch.zeros(10, requires_grad=True))

def layer10(x):
    new_x = x @ layer10_weights + layer10_bias
    return new_x

def softmax(x):
    exp = torch.exp(x)

    sum = torch.sum(exp, dim=1, keepdim=True) 
    #
    # dim=0 表示每列求和，dim=1 表示每行求和
    #
    # keepdim=True 表示不要因为某个维度只剩一个元素就删掉那个维度，会得到类似
    # tensor([[ 6],
    #    [15]])
    # keepdim=False 会得到类似
    # tensor([ 6, 15])

    return exp / sum

def model(x):
    x = flatten(x)
    x = layer512(x)
    x = relu(x)
    x = layer256(x)
    x = relu(x)
    x = layer10(x)
    x = softmax(x)
    return x

def loss_fn(y_pred, y_sample):
    # y_pred: 多个预测结果，每行 10 个概率结果。形如：
    # [
    #     [0.067, 0.108, 0.178, 0.197, 0.035, 0.071, 0.132, 0.049, 0.099, 0.062],
    #     [0.075, 0.082, 0.024, 0.03 , 0.142, 0.115, 0.131, 0.091, 0.115, 0.202],
    #     [0.12 , 0.044, 0.066, 0.108, 0.101, 0.129, 0.099, 0.113, 0.114, 0.107],
    #     [0.038, 0.136, 0.069, 0.147, 0.059, 0.083, 0.075, 0.062, 0.236, 0.135],
    #     [0.114, 0.033, 0.098, 0.097, 0.043, 0.065, 0.124, 0.221, 0.097, 0.108]
    # ]
    #
    # y_sample: 正确答案。是一个一维数组，每一个代表一次预测的正确答案。
    #           因为我们答案是按照0、1、2、3、4、5、6、7、8、9的数字顺序排列的，
    #           所以其数字代表「第几个数字是正确答案」，也刚刚好代表了是答案是什么数字。
    # 形如：
    # [2, 9, 5, 8, 7]
    #

    # 预测的样本数量
    n_samples = len(y_pred)

    # 生成行索引 [0, 1, 2, ..., n_sample-1]，准备为每行抽取出答案对应的概率
    row_indices = range(n_samples)

    #
    # 使用 Python Fancy Indexing 功能
    # 从每行的 10 个概率中，抽取出正确答案对应的那一个概率，我们只关于它觉得正确答案的概率是多少
    # 抽取的结果形如：
    # [0.178, 0.202, 0.129, 0.236, 0.221]
    #
    selected_probs = y_pred[row_indices, y_sample]

    # 为每个答案求负自然对数，求出其最大似然估计，作为 loss 值返回
    # 加上一个特别小的 1e-10 防止 log(0) 得到无穷大
    loss = - torch.log(selected_probs + 1e-10)

    return loss



数据集已加载，共 60000 条样本。
列名: ['image', 'label']


In [2]:
images, labels = preprocess(dataset)

optimizer = torch.optim.SGD([
    layer512_weights, layer512_bias,
    layer256_weights, layer256_bias,
    layer10_weights, layer10_bias
], lr=0.1)

for epoch in range(2000):
    preds = model(images)
    
    loss = loss_fn(preds, labels).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.3025
Epoch 10, Loss: 2.3015
Epoch 20, Loss: 2.3004
Epoch 30, Loss: 2.2992
Epoch 40, Loss: 2.2979
Epoch 50, Loss: 2.2963
Epoch 60, Loss: 2.2943
Epoch 70, Loss: 2.2917
Epoch 80, Loss: 2.2881
Epoch 90, Loss: 2.2830
Epoch 100, Loss: 2.2753
Epoch 110, Loss: 2.2633
Epoch 120, Loss: 2.2436
Epoch 130, Loss: 2.2105
Epoch 140, Loss: 2.1567
Epoch 150, Loss: 2.0784
Epoch 160, Loss: 1.9734
Epoch 170, Loss: 1.8285
Epoch 180, Loss: 1.6409
Epoch 190, Loss: 1.4408
Epoch 200, Loss: 1.2562
Epoch 210, Loss: 1.0996
Epoch 220, Loss: 0.9779
Epoch 230, Loss: 0.8890
Epoch 240, Loss: 0.8240
Epoch 250, Loss: 0.7745
Epoch 260, Loss: 0.7346
Epoch 270, Loss: 0.7009
Epoch 280, Loss: 0.6711
Epoch 290, Loss: 0.6439
Epoch 300, Loss: 0.6186
Epoch 310, Loss: 0.5951
Epoch 320, Loss: 0.5732
Epoch 330, Loss: 0.5530
Epoch 340, Loss: 0.5348
Epoch 350, Loss: 0.5184
Epoch 360, Loss: 0.5037
Epoch 370, Loss: 0.4905
Epoch 380, Loss: 0.4787
Epoch 390, Loss: 0.4680
Epoch 400, Loss: 0.4583
Epoch 410, Loss: 0.4494
Epo

In [3]:
def predict(pil_img):
    # 接收 PIL 图像，返回预测的数字 (0-9)
    
    with torch.no_grad(): # 关闭梯度计算，节省内存并加速
        # 图像预处理
        pixel_data = list(pil_img.getdata())
        img_tensor = torch.tensor(pixel_data, dtype=torch.float32)
        normalized_img = img_tensor / 255.0
        input_batch = normalized_img.unsqueeze(0) # 形状: [1, 784]

        # 前向传播，得到 10 个数字的预测概率
        outputs = model(input_batch)

        # torch.max 返回一个元组 (values, indices)，从 10 个概率中获取最大数的索引，即预测的数字
        _, predicted = torch.max(outputs, 1)
        
        return predicted.item()

test_dataset = load_dataset(
    "parquet", 
    data_files="./mnist_data/mnist/test-00000-of-00001.parquet",
    split="train"
)

test_img = test_dataset[2]["image"]
true_label = test_dataset[2]["label"]
predicted_label = predict(test_img)

print(f"真实标签: {true_label}")
print(f"预测标签: {predicted_label}")

真实标签: 1
预测标签: 1


In [4]:
import random
for idx in random.sample(range(len(test_dataset)), 10):
    test_img = test_dataset[idx]["image"]
    true_label = test_dataset[idx]["label"]
    predicted_label = predict(test_img)

    status = "right" if (true_label == predicted_label) else "wrong"
    print(f"索引 {idx:3d} | 真实: {true_label} | 预测: {predicted_label} | {status}")

索引 8776 | 真实: 1 | 预测: 1 | right
索引 4010 | 真实: 1 | 预测: 1 | right
索引 8511 | 真实: 4 | 预测: 4 | right
索引 7929 | 真实: 6 | 预测: 6 | right
索引 7130 | 真实: 3 | 预测: 3 | right
索引 4670 | 真实: 1 | 预测: 1 | right
索引 5751 | 真实: 7 | 预测: 7 | right
索引 7313 | 真实: 8 | 预测: 8 | right
索引 9959 | 真实: 8 | 预测: 8 | right
索引 5284 | 真实: 4 | 预测: 4 | right
